In [63]:
# from utils import segementation_metrics
from utils import accuracy_metric

ImportError: cannot import name 'accuracy_metric' from 'utils' (c:\Users\user\Documents\GitHub\Korean_Food_Detection\code\tests\utils.py)

In [76]:
import requests
from PIL import Image
import io
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import json
from collections import Counter
import importlib
import utils
importlib.reload(utils)
from utils import segementation_metrics,accuracy_score
import random
import os
from pathlib import Path
import cv2

class FoodDetectionReporter:
    def __init__(self):
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
    def calculate_iou(self, mask1: np.ndarray, mask2: np.ndarray) -> float:
        """Calculate IoU between two binary masks"""
        intersection = np.logical_and(mask1, mask2).sum()
        union = np.logical_or(mask1, mask2).sum()
        return intersection / union if union > 0 else 0

    def calculate_segmentation_metrics(self, pred_masks: List[np.ndarray], 
                                    gt_masks: List[np.ndarray]) -> Dict:
        """Calculate segmentation metrics including IoU"""
        metrics = {
            'iou_scores': [],
            'mean_iou': 0,
            'per_class_iou': {}
        }
        
        # Calculate IoU for each mask pair
        for pred_mask, gt_mask in zip(pred_masks, gt_masks):
            iou = self.calculate_iou(pred_mask, gt_mask)
            metrics['iou_scores'].append(iou)
            
        metrics['mean_iou'] = np.mean(metrics['iou_scores']) if metrics['iou_scores'] else 0
        return metrics

    def generate_iou_plot(self, iou_scores: List[float], labels: List[str]) -> str:
        """Generate IoU distribution plot"""
        plt.figure(figsize=(10, 5))
        plt.bar(labels, iou_scores)
        plt.ylim(0, 1)
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('IoU Score')
        plt.title('Segmentation IoU Scores by Food Item')
        
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        plot_path = os.path.join(output_dir, f'iou_plot_{self.timestamp}.png')
        plt.tight_layout()
        plt.savefig(plot_path)
        plt.close()
        return plot_path
    def analyze_confidence_scores(self, detections: List[Dict]) -> Dict:
        """Analyze confidence scores from detections"""
        confidence_scores = [d['confidence'] for d in detections]
        return {
            'mean': np.mean(confidence_scores),
            'median': np.median(confidence_scores),
            'min': np.min(confidence_scores),
            'max': np.max(confidence_scores),
            'std': np.std(confidence_scores)
        }

    def analyze_class_distribution(self, detections: List[Dict]) -> Dict:
        """Analyze distribution of detected classes"""
        labels = [d['label'] for d in detections]
        class_counts = Counter(labels)
        return dict(class_counts)

    def generate_confidence_plot(self, detections: List[Dict]) -> str:
        """Generate confidence score distribution plot
        
        Args:
            detections: List of detection results with confidence scores
            
        Returns:
            str: Path to saved plot image
        """
        confidence_scores = [d['confidence'] for d in detections]
        labels = [d['label'] for d in detections]
        
        # plt.figure(figsize=(10, 5))
        # plt.bar(labels, confidence_scores)
        # plt.ylim(0, 1)
        # plt.xticks(rotation=45, ha='right')
        # plt.ylabel('Confidence Score')
        # plt.title('Detection Confidence Scores by Food Item')
        
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        plot_path = os.path.join(output_dir, f'confidence_plot_{self.timestamp}.png')
        # plt.tight_layout()
        # plt.savefig(plot_path)
        # plt.close()
        
        return plot_path

    def generate_segmentation_visualization(self, image: np.ndarray, 
                                         masks: List[np.ndarray]) -> str:
        """Generate visualization of segmentation masks"""
        # Create a colored visualization of all masks
        vis_image = image.copy()
        colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0), 
                 (255,0,255), (0,255,255)]  # Add more colors if needed
        
        for mask, color in zip(masks, colors):
            vis_image[mask > 0] = color
            
        output_dir = 'reports'
        vis_path = os.path.join(output_dir, f'segmentation_vis_{self.timestamp}.png')
        cv2.imwrite(vis_path, cv2.cvtColor(vis_image, cv2.COLOR_RGB2BGR))
        return vis_path

    def load_image(self, image_path: str) -> Optional[Image.Image]:
        """Load image from URL or local path"""
        try:
            if image_path.startswith(('http://', 'https://')):
                response = requests.get(image_path)
                response.raise_for_status()
                return Image.open(io.BytesIO(response.content))
            else:
                image_path = os.path.abspath(image_path)
                if not os.path.exists(image_path):
                    raise FileNotFoundError(f"Image file not found: {image_path}")
                return Image.open(image_path)
        except Exception as e:
            print(f"Error loading image: {e}")
            return None

    
        
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        report_path = os.path.join(output_dir, f'detection_report_{self.timestamp}.html')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        return report_path
    def generate_json_report(self, image_path: str, detections: List[Dict], 
                           confidence_plot_path: str, segmentation_metrics: Dict,
                           iou_plot_path: str, segmentation_vis_path: str) -> str:
        """Generate JSON report with analysis results including segmentation metrics
        
        Args:
            image_path: Path to input image
            detections: List of detection results
            confidence_plot_path: Path to confidence plot image
            segmentation_metrics: Dictionary of segmentation metrics
            iou_plot_path: Path to IoU plot image
            segmentation_vis_path: Path to segmentation visualization image
            
        Returns:
            str: Path to saved JSON report
        """
        confidence_stats = self.analyze_confidence_scores(detections)
        class_distribution = self.analyze_class_distribution(detections)
        
        # Convert paths to relative paths
        relative_image_path = os.path.relpath(image_path) if not image_path.startswith(('http://', 'https://')) else image_path
        relative_confidence_path = os.path.relpath(confidence_plot_path)
        relative_iou_plot_path = os.path.relpath(iou_plot_path)
        relative_seg_vis_path = os.path.relpath(segmentation_vis_path)
        segmentation_metric=segementation_metrics()
        accuracy_class= accuracy_score()
        # Create report dictionary
        report_data = {
            "metadata": {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "image_path": relative_image_path
            },
            "detection_results": {
                "detections": [
                    {
                        "label": detection["label"],
                        "confidence": detection["confidence"]
                    } for detection in detections
                ],
                "confidence_statistics": confidence_stats,
                "class_distribution": class_distribution,
                "confidence_plot_path": relative_confidence_path
            },
            "segmentation_results": {
                "metrics": {
                    "mean_iou": float(segmentation_metric["mean_iou"][0]),
                    "best_iou": float(segmentation_metric["best_iou"][0]),
                    "worst_iou": float(segmentation_metric["worst_iou"][0])

                    
                    # "per_class_iou": {
                    #     detection["label"]: iou_score
                    #     for detection, iou_score in zip(detections, segmentation_metrics["iou_scores"])
                    # }
                },
                "visualization_paths": {
                    "iou_plot": relative_iou_plot_path,
                    "segmentation_visualization": relative_seg_vis_path
                },
                "Label_Accuracy": {
                    "accuracy":accuracy_class}
                

                
            }
        }
        
        # Save report
        output_dir = 'reports'
        os.makedirs(output_dir, exist_ok=True)
        report_path = os.path.join(output_dir, f'detection_report_{self.timestamp}.json')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report_data, f, indent=4)
        
        return report_path
def generate_food_detection_report(image_path: str, detections: List[Dict], true_label:List[Dict],
                                 pred_masks: List[np.ndarray], 
                                 gt_masks: List[np.ndarray], 
                                 report_format: str = 'json') -> str:
    """
    Generate food detection report with segmentation metrics
    
    Args:
        image_path: Path to image file or URL
        detections: List of detection results with confidence scores
        pred_masks: List of predicted segmentation masks
        gt_masks: List of ground truth segmentation masks
        report_format: Output format ('json' or 'html'), defaults to 'json'
    """
    reporter = FoodDetectionReporter()
    
    # Load image
    image = reporter.load_image(image_path)
    if image is None:
        return "Error: Could not load image"
    
    # Convert image to numpy array
    image_np = np.array(image)
    
    # Calculate segmentation metrics
    segmentation_metrics = reporter.calculate_segmentation_metrics(pred_masks, gt_masks)
    
    # Generate plots and visualizations
    confidence_plot_path = reporter.generate_confidence_plot(detections)
    iou_plot_path = reporter.generate_iou_plot(
        segmentation_metrics['iou_scores'], 
        [d['label'] for d in detections]
    )
    segmentation_vis_path = reporter.generate_segmentation_visualization(
        image_np, pred_masks
    )
    
    # Generate report based on format
    
    report_path = reporter.generate_json_report(
        image_path, detections, confidence_plot_path,
        segmentation_metrics, iou_plot_path, segmentation_vis_path
    )

    
    return report_path

# Example usage
if __name__ == "__main__":
    image_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_exemples\2-d.jpg"
    
    # Sample detection results
    detections = [
        {'label': 'Fried egg', 'confidence': 0.85},
        {'label': 'Beansprouts', 'confidence': 0.87},
        {'label': 'Cabbage kimchi', 'confidence': 0.81},
        {'label': 'Grilled offal', 'confidence': 0.82}
    ]

    True_labels=[
        {'label': 'Fried egg', 'confidence': 0.85},
        {'label': 'Beansprouts', 'confidence': 0.87},
        {'label': 'Cabbage kimchi', 'confidence': 0.81},
        {'label': 'Grilled offal', 'confidence': 0.82}
    ]
    
    # Create dummy masks for example
    # In practice, these would come from your segmentation model
    image = np.array(Image.open(image_path))
    height, width = image.shape[:2]
    pred_masks = [np.zeros((height, width), dtype=bool) for _ in detections]
    gt_masks = [np.zeros((height, width), dtype=bool) for _ in detections]
    
    report_path = generate_food_detection_report(
        image_path, detections, True_labels,pred_masks, gt_masks,
    )
    print(f"Report generated: {report_path}")

Report generated: reports\detection_report_20241104_093421.json


test for all the folder

In [75]:
import requests
from PIL import Image
import io
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import json
from collections import Counter
from utils import segementation_metrics
import os
from pathlib import Path
import cv2

class BatchFoodDetectionReporter:
    def __init__(self):
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.output_dir = 'reports'
        os.makedirs(self.output_dir, exist_ok=True)
        
    def calculate_iou(self, mask1: np.ndarray, mask2: np.ndarray) -> float:
        """Calculate IoU between two binary masks"""
        intersection = np.logical_and(mask1, mask2).sum()
        union = np.logical_or(mask1, mask2).sum()
        return intersection / union if union > 0 else 0

    def calculate_segmentation_metrics(self, pred_masks: List[np.ndarray], 
                                    gt_masks: List[np.ndarray]) -> Dict:
        """Calculate segmentation metrics including IoU"""
        metrics = {
            'iou_scores': [],
            'mean_iou': 0,
            'per_class_iou': {}
        }
        
        for pred_mask, gt_mask in zip(pred_masks, gt_masks):
            iou = self.calculate_iou(pred_mask, gt_mask)
            metrics['iou_scores'].append(iou)
            
        metrics['mean_iou'] = np.mean(metrics['iou_scores']) if metrics['iou_scores'] else 0
        return metrics

    def generate_aggregate_iou_plot(self, all_iou_scores: Dict[str, List[float]]) -> str:
        """Generate box plot of IoU distributions across all images"""
        plt.figure(figsize=(12, 6))
        plt.boxplot(list(all_iou_scores.values()), labels=list(all_iou_scores.keys()))
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('IoU Score')
        plt.title('IoU Score Distribution by Food Item Across All Images')
        
        plot_path = os.path.join(self.output_dir, f'aggregate_iou_plot_{self.timestamp}.png')
        plt.tight_layout()
        plt.savefig(plot_path)
        plt.close()
        return plot_path

    def analyze_confidence_scores(self, all_detections: List[Dict]) -> Dict:
        """Analyze confidence scores from all detections"""
        confidence_scores = [d['confidence'] for d in all_detections]
        return {
            'mean': float(np.mean(confidence_scores)),
            'median': float(np.median(confidence_scores)),
            'min': float(np.min(confidence_scores)),
            'max': float(np.max(confidence_scores)),
            'std': float(np.std(confidence_scores))
        }

    def generate_confidence_distribution_plot(self, all_detections: List[Dict]) -> str:
        """Generate histogram of confidence scores across all images"""
        confidence_scores = [d['confidence'] for d in all_detections]
        
        plt.figure(figsize=(10, 5))
        plt.hist(confidence_scores, bins=20, range=(0, 1))
        plt.xlabel('Confidence Score')
        plt.ylabel('Count')
        plt.title('Distribution of Confidence Scores Across All Images')
        
        plot_path = os.path.join(self.output_dir, f'confidence_distribution_{self.timestamp}.png')
        plt.tight_layout()
        plt.savefig(plot_path)
        plt.close()
        return plot_path

    def process_single_image(self, image_path: str, detections: List[Dict], 
                           true_labels: List[Dict], pred_masks: List[np.ndarray], 
                           gt_masks: List[np.ndarray]) -> Dict:
        """Process a single image and return its metrics"""
        image = np.array(Image.open(image_path))
        seg_metrics = self.calculate_segmentation_metrics(pred_masks, gt_masks)
        
        return {
            'image_path': image_path,
            'detections': detections,
            'true_labels': true_labels,
            'segmentation_metrics': seg_metrics,
            'confidence_scores': [d['confidence'] for d in detections]
        }

    def generate_batch_report(self, folder_path: str, detection_results: Dict[str, Dict]) -> str:
        """
        Generate comprehensive report for all images in the folder
        
        Args:
            folder_path: Path to folder containing images
            detection_results: Dictionary mapping image paths to their detection results
        """
        # Aggregate metrics across all images
        all_detections = []
        all_iou_scores = {}
        accuracy_metrics = []
        
        for image_path, results in detection_results.items():
            all_detections.extend(results['detections'])
            
            # Aggregate IoU scores by class
            for det, iou in zip(results['detections'], 
                              results['segmentation_metrics']['iou_scores']):
                label = det['label']
                if label not in all_iou_scores:
                    all_iou_scores[label] = []
                all_iou_scores[label].append(iou)
        
        # Generate aggregate plots
        confidence_plot = self.generate_confidence_distribution_plot(all_detections)
        iou_plot = self.generate_aggregate_iou_plot(all_iou_scores)
        
        # Calculate aggregate metrics
        confidence_stats = self.analyze_confidence_scores(all_detections)
        class_distribution = Counter(d['label'] for d in all_detections)
        
        # Create comprehensive report
        report_data = {
            "metadata": {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "folder_path": folder_path,
                "number_of_images": len(detection_results)
            },
            "aggregate_metrics": {
                "confidence_statistics": confidence_stats,
                "class_distribution": dict(class_distribution),
                "plots": {
                    "confidence_distribution": os.path.relpath(confidence_plot),
                    "iou_distribution": os.path.relpath(iou_plot)
                }
            },
            "per_image_results": {
                image_path: {
                    "detections": results['detections'],
                    "true_labels": results['true_labels'],
                    "segmentation_metrics": {
                        "mean_iou": float(results['segmentation_metrics']['mean_iou']),
                        "iou_scores": [float(score) for score in results['segmentation_metrics']['iou_scores']]
                    }
                }
                for image_path, results in detection_results.items()
            }
        }
        
        # Save report
        report_path = os.path.join(self.output_dir, f'batch_detection_report_{self.timestamp}.json')
        with open(report_path, 'w', encoding='utf-8') as f:
            json.dump(report_data, f, indent=4)
        
        return report_path

def process_image_folder(folder_path: str) -> str:
    """
    Process all images in a folder and generate a comprehensive report
    
    Args:
        folder_path: Path to folder containing images
        
    Returns:
        str: Path to the generated report
    """
    reporter = BatchFoodDetectionReporter()
    detection_results = {}
    
    # Process each image in the folder
    for image_file in os.listdir(folder_path):
        if image_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(folder_path, image_file)
            
            # Here you would normally get these from your model
            # This is just placeholder code
            detections = [
                {'label': 'Fried egg', 'confidence': 0.85},
                {'label': 'Beansprouts', 'confidence': 0.87}
            ]
            
            true_labels = [
                {'label': 'Fried egg', 'confidence': 1.0},
                {'label': 'Beansprouts', 'confidence': 1.0}
            ]
            
            # Create dummy masks (replace with actual masks from your model)
            image = np.array(Image.open(image_path))
            height, width = image.shape[:2]
            pred_masks = [np.zeros((height, width), dtype=bool) for _ in detections]
            gt_masks = [np.zeros((height, width), dtype=bool) for _ in detections]
            
            # Process the image
            results = reporter.process_single_image(
                image_path, detections, true_labels, pred_masks, gt_masks
            )
            detection_results[image_path] = results
    
    # Generate and return the batch report
    return reporter.generate_batch_report(folder_path, detection_results)

# Example usage
if __name__ == "__main__":
    folder_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_exemples"
    report_path = process_image_folder(folder_path)
    print(f"Batch report generated: {report_path}")

C:\Users\user\AppData\Local\Temp\ipykernel_25424\3068385733.py:46: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(list(all_iou_scores.values()), labels=list(all_iou_scores.keys()))


Batch report generated: reports\batch_detection_report_20241104_092304.json
